# Embedding-Based Customer Support Insight Platform

This notebook uses a lightweight modern NLP pipeline for 200k support tickets:

```text
Support Tickets
    -> Lightweight Preprocessing
    -> SentenceTransformer Embeddings
    -> FAISS Similarity Search
    -> Response Recommendation
    -> KMeans Issue Clustering
    -> Frustration and Business Insights
```


## 1. Data Preparation

Input columns used by the project:

- Ticket Subject
- Ticket Description
- Resolution
- Ticket Type
- Ticket Priority
- Customer Satisfaction Rating


In [29]:
import re
import string
import warnings

import numpy as np
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 140)

RANDOM_STATE = 42
DATA_PATH = "customer_support_tickets_200k.csv"
MAX_ROWS = 200_000


In [30]:
raw_df = pd.read_csv(DATA_PATH, nrows=MAX_ROWS)
print(raw_df.shape)
raw_df.head()


(200000, 30)


,ticket_id,customer_name,customer_email,product,category,issue_description,resolution_notes,priority,status,channel,...,ticket_resolved_date,escalated,sla_breached,operating_system,browser,payment_method,language,preferred_contact_time,issue_complexity_score,customer_segment
0,1,Patricia Smith,patricia.smith760@outlook.com,Web Portal,Account Suspension,The payment was deducted from my bank account but the transaction shows failed.,Data synchronization restored after backend service restart.,Urgent,Open,Email,...,2023-05-20,No,Yes,MacOS,Edge,PayPal,French,Afternoon,4,Small Business
1,2,Patricia Williams,patricia.williams390@gmail.com,Mobile App,Performance Issue,I found a bug in the latest update affecting report generation.,Provided step-by-step troubleshooting instructions and issue resolved.,Urgent,Closed,Email,...,2024-01-19,Yes,Yes,Windows,Firefox,PayPal,English,Afternoon,2,Small Business
2,3,William Anderson,william.anderson651@outlook.com,Web Portal,Performance Issue,The application crashes whenever I try to upload a file.,Provided step-by-step troubleshooting instructions and issue resolved.,Medium,Closed,Chat,...,2022-12-05,Yes,Yes,Windows,Safari,Bank Transfer,French,Morning,4,Corporate
3,4,David Miller,david.miller672@icloud.com,Payment Gateway,Subscription Cancellation,My subscription was cancelled without my request and I need clarification.,Provided step-by-step troubleshooting instructions and issue resolved.,Medium,Closed,Social Media,...,2024-04-04,Yes,No,Windows,Chrome,Credit Card,Spanish,Afternoon,7,Corporate
4,5,Robert Gonzalez,robert.gonzalez391@hotmail.com,Web Portal,Feature Request,The system is not syncing data across devices properly.,We have reset the account credentials and advised the customer to retry login.,High,Pending Customer,Email,...,2024-08-24,Yes,No,Linux,NaN,Debit Card,Spanish,Evening,3,Corporate


In [31]:
# The 200k dataset uses snake_case names. Rename them into the assignment column names.
column_map = {
    "ticket_id": "Ticket ID",
    "category": "Ticket Type",
    "issue_description": "Ticket Description",
    "resolution_notes": "Resolution",
    "priority": "Ticket Priority",
    "customer_satisfaction_score": "Customer Satisfaction Rating",
    "ticket_created_date": "Ticket Created Date",
    "ticket_resolved_date": "Ticket Resolved Date",
    "status": "Ticket Status",
    "product": "Product Purchased"
}
work = raw_df.rename(columns=column_map).copy()

# Create a short subject from the category and the beginning of the issue text.
work["Ticket Subject"] = (
    work["Ticket Type"].fillna("Support issue").astype(str)
    + " - "
    + work["Ticket Description"].fillna("").astype(str).str.slice(0, 80)
)

input_columns = [
    "Ticket Subject",
    "Ticket Description",
    "Resolution",
    "Ticket Type",
    "Ticket Priority",
    "Customer Satisfaction Rating"
]

work = work[input_columns + [
    "Ticket ID",
    "Ticket Created Date",
    "Ticket Resolved Date",
    "Ticket Status",
    "Product Purchased",
    "resolution_time_hours",
    "escalated",
    "sla_breached"
]].copy()

# Convert numeric columns before any groupby/mean operations.
work["Customer Satisfaction Rating"] = pd.to_numeric(
    work["Customer Satisfaction Rating"],
    errors="coerce"
)
work["resolution_time_hours"] = pd.to_numeric(
    work["resolution_time_hours"],
    errors="coerce"
)

# Convert Yes/No columns to 1/0 so escalation and SLA rates can be averaged.
for column in ["escalated", "sla_breached"]:
    work[column] = (
        work[column]
        .astype(str)
        .str.strip()
        .str.lower()
        .map({"yes": 1, "no": 0, "true": 1, "false": 0, "1": 1, "0": 0})
        .fillna(0)
        .astype(int)
    )

print(work.shape)
work[input_columns + ["resolution_time_hours", "escalated", "sla_breached"]].head()


(200000, 14)


,Ticket Subject,Ticket Description,Resolution,Ticket Type,Ticket Priority,Customer Satisfaction Rating,resolution_time_hours,escalated,sla_breached
0,Account Suspension - The payment was deducted from my bank account but the transaction shows failed.,The payment was deducted from my bank account but the transaction shows failed.,Data synchronization restored after backend service restart.,Account Suspension,Urgent,5,108.36,0,1
1,Performance Issue - I found a bug in the latest update affecting report generation.,I found a bug in the latest update affecting report generation.,Provided step-by-step troubleshooting instructions and issue resolved.,Performance Issue,Urgent,5,87.43,1,1
2,Performance Issue - The application crashes whenever I try to upload a file.,The application crashes whenever I try to upload a file.,Provided step-by-step troubleshooting instructions and issue resolved.,Performance Issue,Medium,5,78.50,1,1
3,Subscription Cancellation - My subscription was cancelled without my request and I need clarification.,My subscription was cancelled without my request and I need clarification.,Provided step-by-step troubleshooting instructions and issue resolved.,Subscription Cancellation,Medium,4,239.36,1,0
4,Feature Request - The system is not syncing data across devices properly.,The system is not syncing data across devices properly.,We have reset the account credentials and advised the customer to retry login.,Feature Request,High,5,122.34,1,0


## 2. Text Construction

The main semantic feature is:

```python
ticket_text = Ticket Subject + Ticket Description
```


In [32]:
work["ticket_text"] = (
    work["Ticket Subject"].fillna("").astype(str)
    + " "
    + work["Ticket Description"].fillna("").astype(str)
)

work[["Ticket Subject", "Ticket Description", "ticket_text"]].head()


,Ticket Subject,Ticket Description,ticket_text
0,Account Suspension - The payment was deducted from my bank account but the transaction shows failed.,The payment was deducted from my bank account but the transaction shows failed.,Account Suspension - The payment was deducted from my bank account but the transaction shows failed. The payment was deducted from my ba...
1,Performance Issue - I found a bug in the latest update affecting report generation.,I found a bug in the latest update affecting report generation.,Performance Issue - I found a bug in the latest update affecting report generation. I found a bug in the latest update affecting report ...
2,Performance Issue - The application crashes whenever I try to upload a file.,The application crashes whenever I try to upload a file.,Performance Issue - The application crashes whenever I try to upload a file. The application crashes whenever I try to upload a file.
3,Subscription Cancellation - My subscription was cancelled without my request and I need clarification.,My subscription was cancelled without my request and I need clarification.,Subscription Cancellation - My subscription was cancelled without my request and I need clarification. My subscription was cancelled wit...
4,Feature Request - The system is not syncing data across devices properly.,The system is not syncing data across devices properly.,Feature Request - The system is not syncing data across devices properly. The system is not syncing data across devices properly.


## 3. Text Preprocessing

Use light preprocessing only:

```text
lowercase -> remove punctuation/special characters -> lemmatization
```

No aggressive stopword removal is used because embedding models already understand context.


In [33]:
import nltk
from nltk.stem import WordNetLemmatizer

# Run once if WordNet is missing:
# nltk.download("wordnet")
# nltk.download("omw-1.4")

lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(token) for token in tokens]
    return " ".join(tokens)

work["clean_ticket_text"] = work["ticket_text"].apply(clean_text)
work[["ticket_text", "clean_ticket_text"]].head()


,ticket_text,clean_ticket_text
0,Account Suspension - The payment was deducted from my bank account but the transaction shows failed. The payment was deducted from my ba...,account suspension the payment wa deducted from my bank account but the transaction show failed the payment wa deducted from my bank acc...
1,Performance Issue - I found a bug in the latest update affecting report generation. I found a bug in the latest update affecting report ...,performance issue i found a bug in the latest update affecting report generation i found a bug in the latest update affecting report gen...
2,Performance Issue - The application crashes whenever I try to upload a file. The application crashes whenever I try to upload a file.,performance issue the application crash whenever i try to upload a file the application crash whenever i try to upload a file
3,Subscription Cancellation - My subscription was cancelled without my request and I need clarification. My subscription was cancelled wit...,subscription cancellation my subscription wa cancelled without my request and i need clarification my subscription wa cancelled without ...
4,Feature Request - The system is not syncing data across devices properly. The system is not syncing data across devices properly.,feature request the system is not syncing data across device properly the system is not syncing data across device properly


## 4. Embedding Generation

Generate dense semantic embeddings using Hugging Face SentenceTransformers.

Recommended model: `all-MiniLM-L6-v2`

Each ticket becomes a 384-dimensional semantic vector.

Install once if needed:

```python
%pip install sentence-transformers
```


In [34]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

texts = work["clean_ticket_text"].tolist()

ticket_embeddings = embedding_model.encode(
    texts,
    batch_size=256,
    show_progress_bar=True,
    normalize_embeddings=True
)

ticket_embeddings = np.asarray(ticket_embeddings, dtype="float32")
print(ticket_embeddings.shape)


Batches:   0%|          | 0/782 [00:00<?, ?it/s]

(200000, 384)


## 5. Vector Storage With FAISS

FAISS stores the 384-dimensional vectors for fast semantic search.

Install once if needed:

```python
%pip install faiss-cpu
```


In [35]:
import faiss

embedding_dim = ticket_embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(embedding_dim)
faiss_index.add(ticket_embeddings)

print("Vectors stored:", faiss_index.ntotal)
print("Embedding dimensions:", embedding_dim)


Vectors stored: 200000
Embedding dimensions: 384


## 6. Similar Ticket Retrieval

Pipeline:

```text
New Ticket -> Embedding -> FAISS Search -> Top-K Similar Tickets
```


In [36]:
def embed_new_ticket(subject, description):
    text = clean_text(str(subject) + " " + str(description))
    embedding = embedding_model.encode(
        [text],
        normalize_embeddings=True
    )
    return np.asarray(embedding, dtype="float32")


def find_similar_tickets(subject, description, top_k=5):
    query_embedding = embed_new_ticket(subject, description)
    scores, indices = faiss_index.search(query_embedding, top_k)

    results = work.iloc[indices[0]].copy()
    results["similarity_score"] = scores[0]
    return results[[
        "Ticket ID",
        "Ticket Subject",
        "Ticket Type",
        "Ticket Priority",
        "Resolution",
        "similarity_score"
    ]]

find_similar_tickets(
    "refund not received",
    "customer says money never came back after refund was approved",
    top_k=5
)


,Ticket ID,Ticket Subject,Ticket Type,Ticket Priority,Resolution,similarity_score
393,394,Refund Request - The payment was deducted from my bank account but the transaction shows failed.,Refund Request,Urgent,Provided step-by-step troubleshooting instructions and issue resolved.,0.644582
390,391,Refund Request - The payment was deducted from my bank account but the transaction shows failed.,Refund Request,Urgent,Subscription status corrected and confirmation email sent to customer.,0.644582
374,375,Refund Request - The payment was deducted from my bank account but the transaction shows failed.,Refund Request,Low,Payment gateway timeout issue fixed and monitoring implemented.,0.644582
137,138,Refund Request - The payment was deducted from my bank account but the transaction shows failed.,Refund Request,Urgent,Data synchronization restored after backend service restart.,0.644582
15,16,Refund Request - The payment was deducted from my bank account but the transaction shows failed.,Refund Request,High,Security settings updated and customer notified of precautionary measures.,0.644582


## 7. Suggested Response Recommendation

Use retrieved tickets and their previous resolutions as a lightweight retrieval-based support assistant.


In [37]:
def recommend_response(subject, description, top_k=5):
    similar = find_similar_tickets(subject, description, top_k=top_k)
    resolved = similar[similar["Resolution"].notna() & (similar["Resolution"].astype(str).str.len() > 0)]

    if len(resolved) == 0:
        suggestion = "Escalate this ticket because no close historical resolution was found."
    else:
        suggestion = resolved.iloc[0]["Resolution"]

    return {
        "suggested_response": suggestion,
        "similar_tickets": similar
    }

example_response = recommend_response(
    "refund not received",
    "customer says money never came back after refund was approved",
    top_k=5
)

print(example_response["suggested_response"])
example_response["similar_tickets"]


Provided step-by-step troubleshooting instructions and issue resolved.


,Ticket ID,Ticket Subject,Ticket Type,Ticket Priority,Resolution,similarity_score
393,394,Refund Request - The payment was deducted from my bank account but the transaction shows failed.,Refund Request,Urgent,Provided step-by-step troubleshooting instructions and issue resolved.,0.644582
390,391,Refund Request - The payment was deducted from my bank account but the transaction shows failed.,Refund Request,Urgent,Subscription status corrected and confirmation email sent to customer.,0.644582
374,375,Refund Request - The payment was deducted from my bank account but the transaction shows failed.,Refund Request,Low,Payment gateway timeout issue fixed and monitoring implemented.,0.644582
137,138,Refund Request - The payment was deducted from my bank account but the transaction shows failed.,Refund Request,Urgent,Data synchronization restored after backend service restart.,0.644582
15,16,Refund Request - The payment was deducted from my bank account but the transaction shows failed.,Refund Request,High,Security settings updated and customer notified of precautionary measures.,0.644582


## 8. Recurring Issue Detection

Cluster embeddings to identify common customer problems.

This notebook uses KMeans because it is simple and scalable for 200k rows.


In [38]:
N_CLUSTERS = 12

cluster_model = KMeans(
    n_clusters=N_CLUSTERS,
    random_state=RANDOM_STATE,
    n_init=10
)

work["issue_cluster"] = cluster_model.fit_predict(ticket_embeddings)

vectorizer = CountVectorizer(max_features=3000, stop_words="english", ngram_range=(1, 2))
term_matrix = vectorizer.fit_transform(work["clean_ticket_text"])
terms = np.array(vectorizer.get_feature_names_out())


def top_terms_for_cluster(cluster_id, top_n=8):
    mask = work["issue_cluster"].values == cluster_id
    mean_terms = np.asarray(term_matrix[mask].mean(axis=0)).ravel()
    top_indices = mean_terms.argsort()[::-1][:top_n]
    return ", ".join(terms[top_indices])

cluster_summary = (
    work.groupby("issue_cluster")
    .agg(
        ticket_count=("Ticket ID", "count"),
        top_ticket_type=("Ticket Type", lambda x: x.mode().iat[0]),
        avg_satisfaction=("Customer Satisfaction Rating", "mean"),
        avg_resolution_hours=("resolution_time_hours", "mean"),
        escalation_rate=("escalated", "mean"),
        sla_breach_rate=("sla_breached", "mean")
    )
    .reset_index()
)

cluster_summary["top_terms"] = cluster_summary["issue_cluster"].apply(top_terms_for_cluster)
cluster_summary.sort_values("ticket_count", ascending=False).head(12)


,issue_cluster,ticket_count,top_ticket_type,avg_satisfaction,avg_resolution_hours,escalation_rate,sla_breach_rate,top_terms
4,4,20232,Subscription Cancellation,3.002867,120.475418,0.506623,0.495898,"statement, billing, discrepancy, discrepancy billing, statement month, month, billing statement, month discrepancy"
5,5,20126,Security Concern,2.998211,119.347370,0.502832,0.501491,"payment, account, wa deducted, bank account, transaction, transaction failed, payment wa, failed"
6,6,19988,Login Issue,2.979588,120.955456,0.496748,0.499700,"authentication code, delivered, code, delivered phone, code delivered, factor authentication, factor, phone"
1,1,19978,Feature Request,3.000250,120.564933,0.494494,0.497948,"request, refund, like request, charge, like, recent, recent charge, refund recent"
7,7,19975,Bug Report,3.025832,120.884397,0.503279,0.496521,"performance, experiencing slow, using dashboard, using, dashboard, performance using, slow, experiencing"
9,9,19942,Bug Report,3.008124,121.280016,0.505165,0.504112,"data, device properly, device, data device, syncing, syncing data, properly, properly syncing"
8,8,19912,Subscription Cancellation,2.995781,121.191698,0.500301,0.497640,"request, subscription, cancelled request, subscription wa, wa, request need, need, wa cancelled"
3,3,19907,Refund Request,2.996333,120.339392,0.508515,0.507058,"report, bug, generation, report generation, update affecting, update, affecting, affecting report"
2,2,19883,Feature Request,3.002012,120.307434,0.499472,0.501836,"crash try, try upload, crash, application crash, upload file, upload, application, file"
11,11,13976,Security Concern,3.006583,120.127939,0.504937,0.501646,"access, credential, access account, account, account entering, correct credential, correct, entering"


## 9. Sentiment / Frustration Analysis

Use satisfaction ratings and operational signals to identify frustrated customers, escalation risk, and churn indicators.


In [39]:
def rating_to_sentiment(rating):
    if pd.isna(rating):
        return "neutral"
    if rating <= 2:
        return "negative"
    if rating == 3:
        return "neutral"
    return "positive"

priority_risk = {
    "Low": 1,
    "Medium": 2,
    "High": 3,
    "Critical": 4
}

work["sentiment"] = work["Customer Satisfaction Rating"].apply(rating_to_sentiment)
work["priority_score"] = work["Ticket Priority"].map(priority_risk).fillna(2)
work["frustration_score"] = (
    (5 - work["Customer Satisfaction Rating"].fillna(3)) * 2
    + work["priority_score"]
    + work["escalated"].astype(int) * 2
    + work["sla_breached"].astype(int) * 2
)

work["frustration_level"] = pd.cut(
    work["frustration_score"],
    bins=[-1, 5, 9, 20],
    labels=["low", "medium", "high"]
)

work[[
    "Ticket ID",
    "Ticket Type",
    "Ticket Priority",
    "Customer Satisfaction Rating",
    "sentiment",
    "frustration_score",
    "frustration_level"
]].head()


,Ticket ID,Ticket Type,Ticket Priority,Customer Satisfaction Rating,sentiment,frustration_score,frustration_level
0,1,Account Suspension,Urgent,5,positive,4.0,low
1,2,Performance Issue,Urgent,5,positive,6.0,medium
2,3,Performance Issue,Medium,5,positive,6.0,medium
3,4,Subscription Cancellation,Medium,4,positive,6.0,medium
4,5,Feature Request,High,5,positive,5.0,low


## 10. Business Insight Generation

Generate insights for:

- most common issue
- increasing complaint cluster
- high-frustration categories
- slowest resolutions


In [40]:
work["Ticket Created Date"] = pd.to_datetime(work["Ticket Created Date"], errors="coerce")
work["month"] = work["Ticket Created Date"].dt.to_period("M").astype(str)

most_common_issues = cluster_summary.sort_values("ticket_count", ascending=False)

high_frustration_categories = (
    work.groupby("Ticket Type")
    .agg(
        tickets=("Ticket ID", "count"),
        avg_frustration=("frustration_score", "mean"),
        high_frustration_rate=("frustration_level", lambda x: (x == "high").mean()),
        avg_satisfaction=("Customer Satisfaction Rating", "mean")
    )
    .reset_index()
    .sort_values("avg_frustration", ascending=False)
)

slowest_resolutions = (
    work.groupby("Ticket Type")
    .agg(
        tickets=("Ticket ID", "count"),
        avg_resolution_hours=("resolution_time_hours", "mean"),
        sla_breach_rate=("sla_breached", "mean")
    )
    .reset_index()
    .sort_values("avg_resolution_hours", ascending=False)
)

monthly_cluster_counts = (
    work.groupby(["month", "issue_cluster"])
    .size()
    .reset_index(name="tickets")
    .sort_values(["issue_cluster", "month"])
)
monthly_cluster_counts["previous_month_tickets"] = monthly_cluster_counts.groupby("issue_cluster")["tickets"].shift(1)
monthly_cluster_counts["mom_change"] = monthly_cluster_counts["tickets"] - monthly_cluster_counts["previous_month_tickets"]

latest_month = monthly_cluster_counts["month"].max()
increasing_complaint_clusters = (
    monthly_cluster_counts[monthly_cluster_counts["month"] == latest_month]
    .merge(cluster_summary, on="issue_cluster", how="left")
    .sort_values("mom_change", ascending=False)
)

print("Most common issues")
display(most_common_issues.head(10))

print("Increasing complaint clusters")
display(increasing_complaint_clusters.head(10))

print("High-frustration categories")
display(high_frustration_categories.head(10))

print("Slowest resolutions")
display(slowest_resolutions.head(10))


Most common issues


,issue_cluster,ticket_count,top_ticket_type,avg_satisfaction,avg_resolution_hours,escalation_rate,sla_breach_rate,top_terms
4,4,20232,Subscription Cancellation,3.002867,120.475418,0.506623,0.495898,"statement, billing, discrepancy, discrepancy billing, statement month, month, billing statement, month discrepancy"
5,5,20126,Security Concern,2.998211,119.347370,0.502832,0.501491,"payment, account, wa deducted, bank account, transaction, transaction failed, payment wa, failed"
6,6,19988,Login Issue,2.979588,120.955456,0.496748,0.499700,"authentication code, delivered, code, delivered phone, code delivered, factor authentication, factor, phone"
1,1,19978,Feature Request,3.000250,120.564933,0.494494,0.497948,"request, refund, like request, charge, like, recent, recent charge, refund recent"
7,7,19975,Bug Report,3.025832,120.884397,0.503279,0.496521,"performance, experiencing slow, using dashboard, using, dashboard, performance using, slow, experiencing"
9,9,19942,Bug Report,3.008124,121.280016,0.505165,0.504112,"data, device properly, device, data device, syncing, syncing data, properly, properly syncing"
8,8,19912,Subscription Cancellation,2.995781,121.191698,0.500301,0.497640,"request, subscription, cancelled request, subscription wa, wa, request need, need, wa cancelled"
3,3,19907,Refund Request,2.996333,120.339392,0.508515,0.507058,"report, bug, generation, report generation, update affecting, update, affecting, affecting report"
2,2,19883,Feature Request,3.002012,120.307434,0.499472,0.501836,"crash try, try upload, crash, application crash, upload file, upload, application, file"
11,11,13976,Security Concern,3.006583,120.127939,0.504937,0.501646,"access, credential, access account, account, account entering, correct credential, correct, entering"


Increasing complaint clusters


,month,issue_cluster,tickets,previous_month_tickets,mom_change,ticket_count,top_ticket_type,avg_satisfaction,avg_resolution_hours,escalation_rate,sla_breach_rate,top_terms
6,2024-12,6,595,531.0,64.0,19988,Login Issue,2.979588,120.955456,0.496748,0.499700,"authentication code, delivered, code, delivered phone, code delivered, factor authentication, factor, phone"
11,2024-12,11,439,378.0,61.0,13976,Security Concern,3.006583,120.127939,0.504937,0.501646,"access, credential, access account, account, account entering, correct credential, correct, entering"
4,2024-12,4,602,547.0,55.0,20232,Subscription Cancellation,3.002867,120.475418,0.506623,0.495898,"statement, billing, discrepancy, discrepancy billing, statement month, month, billing statement, month discrepancy"
2,2024-12,2,568,518.0,50.0,19883,Feature Request,3.002012,120.307434,0.499472,0.501836,"crash try, try upload, crash, application crash, upload file, upload, application, file"
8,2024-12,8,580,531.0,49.0,19912,Subscription Cancellation,2.995781,121.191698,0.500301,0.497640,"request, subscription, cancelled request, subscription wa, wa, request need, need, wa cancelled"
9,2024-12,9,597,553.0,44.0,19942,Bug Report,3.008124,121.280016,0.505165,0.504112,"data, device properly, device, data device, syncing, syncing data, properly, properly syncing"
5,2024-12,5,557,540.0,17.0,20126,Security Concern,2.998211,119.347370,0.502832,0.501491,"payment, account, wa deducted, bank account, transaction, transaction failed, payment wa, failed"
7,2024-12,7,569,552.0,17.0,19975,Bug Report,3.025832,120.884397,0.503279,0.496521,"performance, experiencing slow, using dashboard, using, dashboard, performance using, slow, experiencing"
0,2024-12,0,59,50.0,9.0,2009,Data Sync Issue,3.031857,118.499791,0.500747,0.514186,"access, access account, account, account entering, correct, entering correct, entering, correct credential"
1,2024-12,1,535,527.0,8.0,19978,Feature Request,3.000250,120.564933,0.494494,0.497948,"request, refund, like request, charge, like, recent, recent charge, refund recent"


High-frustration categories


,Ticket Type,tickets,avg_frustration,high_frustration_rate,avg_satisfaction
8,Security Concern,20040,8.024102,0.356487,2.993513
4,Login Issue,20002,8.022848,0.356764,2.999000
6,Performance Issue,20074,8.021122,0.351549,2.995467
0,Account Suspension,19864,8.021043,0.350685,2.987213
1,Bug Report,19981,8.018267,0.352485,3.000500
3,Feature Request,20169,8.005999,0.354901,2.996034
5,Payment Problem,19997,7.996299,0.349952,3.007701
7,Refund Request,19900,7.994020,0.347990,3.010704
2,Data Sync Issue,19877,7.988077,0.346280,3.008502
9,Subscription Cancellation,20096,7.957355,0.345094,3.012042


Slowest resolutions


,Ticket Type,tickets,avg_resolution_hours,sla_breach_rate
2,Data Sync Issue,19877,121.021393,0.502339
5,Payment Problem,19997,120.939464,0.499725
6,Performance Issue,20074,120.728006,0.507572
4,Login Issue,20002,120.661216,0.500600
3,Feature Request,20169,120.610610,0.497347
9,Subscription Cancellation,20096,120.543078,0.494626
0,Account Suspension,19864,120.414983,0.499094
8,Security Concern,20040,120.362533,0.499052
1,Bug Report,19981,120.318886,0.502577
7,Refund Request,19900,119.794275,0.499246


## Export Outputs


In [41]:
ticket_predictions = work[[
    "Ticket ID",
    "Ticket Subject",
    "Ticket Type",
    "Ticket Priority",
    "Customer Satisfaction Rating",
    "sentiment",
    "frustration_level",
    "issue_cluster"
]]

ticket_predictions.to_csv("ticket_predictions.csv", index=False)
cluster_summary.to_csv("recurring_issue_clusters.csv", index=False)
high_frustration_categories.to_csv("high_frustration_categories.csv", index=False)
slowest_resolutions.to_csv("slowest_resolutions.csv", index=False)
increasing_complaint_clusters.to_csv("increasing_complaint_clusters.csv", index=False)

print("Saved outputs successfully.")


Saved outputs successfully.
